In [ ]:
#MINI PROJET

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

print("="*50)
print("MINI-PROJET: ANALYSE MARKETING")
print("="*50)

np.random.seed(42)

# Données simplifiées
n = 2000
states = ['California', 'New York', 'Texas', 'Florida', 'Illinois']
cities = ['Los Angeles', 'NYC', 'Houston', 'Miami', 'Chicago', 'San Francisco', 'Dallas']
products = ['Laptop', 'Phone', 'Chair', 'Table', 'Pen', 'Paper']

df = pd.DataFrame({
    'Customer': [f'Client_{i}' for i in range(1, 501)] * 4,
    'State': np.random.choice(states, n),
    'City': np.random.choice(cities, n),
    'Product': np.random.choice(products, n),
    'Sales': np.random.uniform(50, 1000, n),
    'Profit': np.random.uniform(-20, 200, n),
    'Discount': np.random.choice([0, 0.1, 0.2], n, p=[0.7, 0.2, 0.1])
})

print(f"Dataset: {len(df)} transactions, {len(df['Customer'].unique())} clients")
print(f"Période: Données simulées\n")

print("="*40)
print("1️ÉTATS LES PLUS PERFORMANTS")
print("="*40)

state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=False)

# Graphique
plt.bar(state_sales.index, state_sales.values, color='steelblue', edgecolor='black')
plt.title('Ventes totales par État', fontsize=14, fontweight='bold')
plt.xlabel('État')
plt.ylabel('Ventes ($)')
plt.xticks(rotation=45)
for i, v in enumerate(state_sales.values):
    plt.text(i, v + 5000, f'${v:,.0f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"🏆 Top 3 États:")
for i, (state, sales) in enumerate(state_sales.head(3).items(), 1):
    print(f"   {i}. {state}: ${sales:,.0f}")

print("\n" + "="*40)
print("2️COMPARAISON NY vs CALIFORNIE")
print("="*40)

ny_sales = df[df['State']=='New York']['Sales'].sum()
ca_sales = df[df['State']=='California']['Sales'].sum()
ny_profit = df[df['State']=='New York']['Profit'].sum()
ca_profit = df[df['State']=='California']['Profit'].sum()

# Graphique
x = ['New York', 'California']
sales = [ny_sales, ca_sales]
profits = [ny_profit, ca_profit]

x_pos = range(len(x))
plt.bar([i - 0.2 for i in x_pos], sales, width=0.4, label='Ventes', color='steelblue')
plt.bar([i + 0.2 for i in x_pos], profits, width=0.4, label='Profits', color='coral')
plt.xticks(x_pos, x)
plt.title('Comparaison New York vs Californie', fontsize=14, fontweight='bold')
plt.ylabel('Montant ($)')
plt.legend()
for i, v in enumerate(sales):
    plt.text(i - 0.2, v + 5000, f'${v:,.0f}', ha='center', fontweight='bold', fontsize=9)
for i, v in enumerate(profits):
    plt.text(i + 0.2, v + 5000, f'${v:,.0f}', ha='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.show()

print(f"New York: Ventes=${ny_sales:,.0f} | Profit=${ny_profit:,.0f}")
print(f"Californie: Ventes=${ca_sales:,.0f} | Profit=${ca_profit:,.0f}")
print(f"Différence: ${abs(ny_sales-ca_sales):,.0f}")

print("\n" + "="*40)
print("3️CLIENT EXCEPTIONNEL À NEW YORK")
print("="*40)

ny_customers = df[df['State']=='New York'].groupby('Customer')['Profit'].sum().sort_values(ascending=False)
top_customer = ny_customers.index[0]
top_profit = ny_customers.values[0]

print(f"Client exceptionnel: {top_customer}")
print(f"Profit généré: ${top_profit:,.0f}")

print("\n" + "="*40)
print("4️RENTABILITÉ PAR ÉTAT")
print("="*40)

state_profit_margin = df.groupby('State').apply(lambda x: (x['Profit'].sum() / x['Sales'].sum()) * 100).sort_values()

colors = ['green' if x > 0 else 'red' for x in state_profit_margin.values]
plt.barh(state_profit_margin.index, state_profit_margin.values, color=colors, edgecolor='black')
plt.title('Marge bénéficiaire par État', fontsize=14, fontweight='bold')
plt.xlabel('Marge (%)')
plt.axvline(x=0, color='black', linestyle='-', alpha=0.5)
for i, v in enumerate(state_profit_margin.values):
    plt.text(v + 0.5, i, f'{v:.1f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Meilleure marge: {state_profit_margin.idxmax()} ({state_profit_margin.max():.1f}%)")
print(f"Pire marge: {state_profit_margin.idxmin()} ({state_profit_margin.min():.1f}%)")

print("\n" + "="*40)
print("5️PRINCIPE DE PARETO (80/20)")
print("="*40)

customer_profit = df.groupby('Customer')['Profit'].sum().sort_values(ascending=False)
cumsum_profit = customer_profit.cumsum()
total_profit = cumsum_profit.iloc[-1]
pct_customers = (cumsum_profit / total_profit) * 100

# Trouver combien de clients font 80% du profit
n_80 = (pct_customers <= 80).sum() + 1
pct_clients = (n_80 / len(customer_profit)) * 100

# Courbe de Pareto
plt.plot(range(1, len(pct_customers)+1), pct_customers.values, linewidth=2, color='steelblue')
plt.axhline(y=80, color='red', linestyle='--', label='Seuil 80%')
plt.axvline(x=n_80, color='green', linestyle='--', label=f'{pct_clients:.1f}% des clients')
plt.title('Courbe de Pareto: Clients vs Profit', fontsize=14, fontweight='bold')
plt.xlabel('Clients cumulés (%)')
plt.ylabel('Profit cumulé (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{n_80} clients génèrent 80% du profit")
print(f"Soit {pct_clients:.1f}% des clients")
if pct_clients <= 25:
    print("✅ Le principe 80/20 s'applique!")
else:
    print("⚠️ Le principe 80/20 s'applique partiellement")

print("\n" + "="*40)
print("6️⃣ TOP 20 VILLES")
print("="*40)

city_sales = df.groupby('City')['Sales'].sum().sort_values(ascending=False).head(20)
city_profit = df.groupby('City')['Profit'].sum().sort_values(ascending=False).head(20)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.barh(range(len(city_sales)), city_sales.values, color='steelblue', edgecolor='black')
plt.yticks(range(len(city_sales)), city_sales.index)
plt.title('Top 20 Villes par Ventes')
plt.xlabel('Ventes ($)')

plt.subplot(1, 2, 2)
colors_profit = ['green' if x > 0 else 'red' for x in city_profit.values]
plt.barh(range(len(city_profit)), city_profit.values, color=colors_profit, edgecolor='black')
plt.yticks(range(len(city_profit)), city_profit.index)
plt.title('Top 20 Villes par Profit')
plt.xlabel('Profit ($)')

plt.tight_layout()
plt.show()

print(f"🏆 Ville avec plus de ventes: {city_sales.index[0]} (${city_sales.values[0]:,.0f})")
print(f"💰 Ville avec plus de profit: {city_profit.index[0]} (${city_profit.values[0]:,.0f})")

print("\n" + "="*40)
print("7️TOP 20 CLIENTS PAR VENTES")
print("="*40)

top_clients_sales = df.groupby('Customer')['Sales'].sum().sort_values(ascending=False).head(20)

plt.barh(range(len(top_clients_sales)), top_clients_sales.values, color='coral', edgecolor='black')
plt.yticks(range(len(top_clients_sales)), top_clients_sales.index, fontsize=8)
plt.title('Top 20 Clients par Ventes', fontsize=14, fontweight='bold')
plt.xlabel('Ventes ($)')
plt.tight_layout()
plt.show()

print("🏆 Top 5 clients:")
for i, (client, sales) in enumerate(top_clients_sales.head(5).items(), 1):
    print(f"   {i}. {client}: ${sales:,.0f}")

# Pareto pour les ventes
print("\n" + "="*40)
print("8️PARETO: Clients vs Ventes")
print("="*40)

customer_sales = df.groupby('Customer')['Sales'].sum().sort_values(ascending=False)
cumsum_sales = customer_sales.cumsum()
total_sales = cumsum_sales.iloc[-1]
pct_sales = (cumsum_sales / total_sales) * 100
n_80_sales = (pct_sales <= 80).sum() + 1
pct_clients_sales = (n_80_sales / len(customer_sales)) * 100

# Courbe de Pareto pour ventes
plt.plot(range(1, len(pct_sales)+1), pct_sales.values, linewidth=2, color='coral')
plt.axhline(y=80, color='red', linestyle='--', label='Seuil 80%')
plt.axvline(x=n_80_sales, color='green', linestyle='--', label=f'{pct_clients_sales:.1f}% des clients')
plt.title('Courbe de Pareto: Clients vs Ventes', fontsize=14, fontweight='bold')
plt.xlabel('Clients cumulés (%)')
plt.ylabel('Ventes cumulées (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{n_80_sales} clients génèrent 80% des ventes")
print(f"Soit {pct_clients_sales:.1f}% des clients")

print("\n" + "="*50)
print("RECOMMANDATIONS MARKETING")
print("="*50)

top_states_strategic = state_sales.head(2).index.tolist()
top_cities_strategic = city_sales.head(3).index.tolist()

print(f"""
 PRIORITÉ GÉOGRAPHIQUE:
   • États prioritaires: {', '.join(top_states_strategic)}
   • Villes prioritaires: {', '.join(top_cities_strategic)}

 GESTION CLIENTS:
   • Fidéliser les top 20% clients (80% du profit)
   • Programme VIP pour {top_customer}

 OPTIMISATION:
   • Éviter les remises excessives (>20%)
   • Renforcer la présence dans les villes rentables

 PROCHAINES ACTIONS:
   1. Campagne marketing ciblée sur {top_states_strategic[0]}
   2. Programme de fidélisation pour top clients
   3. Analyse des produits à marge négative
""")